In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
xmlFile='C:/3S/Bochum_WV.xml' # r'C:\3S\Bochum_WV.xml'
tree = ET.parse(xmlFile) # xml.etree.ElementTree.ElementTree

In [3]:
root = tree.getroot() # xml.etree.ElementTree.Element

In [4]:
pm = {c:p for p in root.iter() for c in p}  

In [5]:
tableNames=[]
oldTableName=None
for element in root.iter():
    p = None
    if element in pm:
        p = pm[element]
    if p != root:
        continue
    actTableName=element.tag
    if actTableName != oldTableName:
        tableNames.append(actTableName)
        oldTableName=actTableName
print(tableNames)
    

['AGSN', 'ALLG', 'ALLG_BZ', 'ARRW', 'ATMO', 'AVOS', 'AVOS_ROWS', 'BREF', 'BZAG', 'BZAG_BZ', 'CIRC', 'CONT', 'CRGL', 'DATENEBENE', 'DPGR', 'DPGR_BZ', 'DPGR_ROWS', 'DRNP', 'DTRO', 'DTRO_ROWD', 'EBES', 'EBES_BZ', 'ELEMENTQUERY', 'ETAM', 'ETAM_ROWS', 'ETAR', 'ETAR_ROWS', 'ETAU', 'ETAU_ROWS', 'FSTF', 'FWBZ', 'GRAV', 'GTXT', 'HAUS', 'HYDR', 'HYDR_BZ', 'KLAP', 'KLAP_BZ', 'KNOT', 'KNOT_BZ', 'LAYR', 'LFAL', 'LFAL_BZ', 'LFKT', 'LFKT_ROWT', 'LTGR', 'MAPG', 'MAPG_ROWS', 'MODELL', 'NRCV', 'NSCH', 'NSCH_BZ', 'PARI', 'PARI_BZ', 'PARV', 'PARZ', 'PARZ_BZ', 'PGPR', 'PGRP', 'PGRP_BZ', 'PGRP_PUMP', 'PGRP_PUMP_BZ', 'PHI1', 'PHI1_ROWT', 'PHI2', 'PHI2_ROWS', 'PHIV', 'PHIV_ROWS', 'POLY', 'PROZESSE', 'PUMD', 'PUMD_ROWT', 'PUMK', 'PUMK_ROWS', 'PUMP', 'PUMP_BZ', 'PVAR', 'PVAR_ROWT', 'PZON', 'PZVR', 'PZVR_BZ', 'QVAR', 'QVAR_ROWT', 'RADD', 'RADD_BZ', 'RART', 'RART_BZ', 'RCON', 'RDIV', 'RDIV_BZ', 'RECT', 'REGP', 'REGV', 'REGV_BZ', 'RHYS', 'RHYS_BZ', 'RLVG', 'RLVG_BZ', 'RMES', 'RMES_BZ', 'RMES_DPTS', 'RMUL', 'RMUL_B

In [6]:
dataFrames={}
for tableName in tableNames:
    all_records = []
    for elementRow in root.iter(tag=tableName):
        record = {}
        for elementCol in elementRow:
            record[elementCol.tag] = elementCol.text
        all_records.append(record)
    dataFrames[tableName]=pd.DataFrame(all_records) 

In [7]:
dataFrames['SWVT_ROWT'].ZEIT=dataFrames['SWVT_ROWT'].ZEIT.str.replace(',', '.')
dataFrames['SWVT_ROWT'].W=dataFrames['SWVT_ROWT'].W.str.replace(',', '.')

In [8]:
dataFrames['SWVT_ROWT'].ZEIT=pd.to_numeric(dataFrames['SWVT_ROWT'].ZEIT,errors='coerce') 
dataFrames['SWVT_ROWT'].W=pd.to_numeric(dataFrames['SWVT_ROWT'].W,errors='coerce') 

In [9]:
dfSWVT=pd.merge(dataFrames['SWVT_ROWT'],dataFrames['SWVT'],left_on='fk',right_on='pk')

In [10]:
dfSWVT=dfSWVT.assign(_TimeRank=dfSWVT.groupby(['NAME'])['ZEIT'].rank(method='first'))
dfSWVT.dtypes

W               float64
ZEIT            float64
fk               object
pk_x             object
BESCHREIBUNG     object
IDREFERENZ       object
INTPOL           object
NAME             object
ZEITOPTION       object
fkDE             object
pk_y             object
_TimeRank       float64
dtype: object

In [12]:
dfSWVT[dfSWVT['NAME'] =='AHON_PUZL']

,W,ZEIT,fk,pk_x,BESCHREIBUNG,IDREFERENZ,INTPOL,NAME,ZEITOPTION,fkDE,pk_y,_TimeRank
136,2.807703,NaN,5649099660669529924,5498721482270951061,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,NaN
137,2.876065,46.0,5649099660669529924,5722834176172694024,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,1.0
138,2.871181,700.0,5649099660669529924,4620599462270755687,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,2.0
139,2.695395,1420.0,5649099660669529924,4658891157033655741,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,3.0
140,2.685629,2500.0,5649099660669529924,5379511738592242286,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,4.0
141,2.744224,3759.0,5649099660669529924,5454182269429558563,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,5.0
142,2.729576,5379.0,5649099660669529924,5552532866067923221,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,6.0
143,6.660000,66666.0,5649099660669529924,5080232238662339765,Typ=P~Einheit=bar~86DatenpunktID=122001302~86D...,BO W 2001AHON UZL PUZL,0,AHON_PUZL,0,4808644906834962875,5649099660669529924,7.0
